# Model Architecture - ChatKasir v0
- ID: CACC715D6Y0970
- Nama: Achmad Rif'an
- Email: CACC715D6Y0970@student.devacademy.id
- ID Tim Capstone: CC26-PSU065
- Email Google Group: CC26-PSU065@student.devacademy.id

## 1. Imports Library

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

# verifikasi versi
print(f"TensorFlow: {tf.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

# check GPU yabg tersedia
print(f"GPU tersedia: {len(tf.config.list_physical_devices('GPU')) > 0}")

TensorFlow: 2.21.0
NumPy: 2.4.4
Pandas: 3.0.2
GPU tersedia: False


## 2. Eksplorasi Awal Dataset Food

In [2]:
# load dataset
url_food_utama = "https://drive.google.com/uc?id=1xpoFjqAT9K0uwzSpVADm_EfKqG7dxVUI"

df_food = pd.read_csv(url_food_utama)

# cek ukuran dataset food
print(df_food.shape)

(18558, 1)


In [3]:
# cek 10 baris awal data
df_food.head(10)

,name
0,abon
1,abon ayam
2,abon burger
3,abon cheese burger
4,abon goreng ayam
5,abon goreng sapi
6,abon gulung
7,abon haruwan
8,abon keju
9,abon pedas


In [4]:
# cek tipe data per kolom
df_food.info()

<class 'pandas.DataFrame'>
RangeIndex: 18558 entries, 0 to 18557
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   name    18558 non-null  str  
dtypes: str(1)
memory usage: 145.1 KB


In [5]:
# cek nilai kosong
df_food.isnull().sum()


name    0
dtype: int64

## 3. Eksplorasi Awal Dataset Slang

In [6]:
# load dataset
url_slang_utama = "https://drive.google.com/uc?id=1G14C1qcqOp06Xs1HFiorE3Us_LLtaBs7"

df_slang = pd.read_csv(url_slang_utama)

# cek ukuran dataset slang
print(df_slang.shape)

(1231, 2)


In [7]:
# cek 10 baris awal data
df_slang.head(10)

,slang,formal
0,aa,kakak
1,abanggg,abang
2,abanggku,abangku
3,abeees,habis
4,abes,habis
5,abez,habis
6,abg,abang
7,abgnya,abangnya
8,abgnyaah,abangnya
9,abiez,habis


In [8]:
# cek tipe data per kolom
df_slang.info()

<class 'pandas.DataFrame'>
RangeIndex: 1231 entries, 0 to 1230
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   slang   1231 non-null   str  
 1   formal  1231 non-null   str  
dtypes: str(2)
memory usage: 19.4 KB


In [9]:
# cek nilai kosong
df_slang.isnull().sum()

slang     0
formal    0
dtype: int64

## 4. Eksplorasi Awal Dataset Sintetis 10000

In [10]:
# load dataset
url_synthetic_10000 = "https://drive.google.com/uc?id=15luIrYGJEZpbH-Wf7foXHXo3LBt0MBqU"

df_synthetic = pd.read_csv(url_synthetic_10000)

# cek ukuran dataset sintetis
print(df_synthetic.shape)

(10050, 5)


In [11]:
# cek 10 baris awal data
df_synthetic.head(10)

,input_text,product,quantity,price_satuan,pattern
0,minta 7 nasi kari limasari nasi putih ya [SEP]...,nasi kari limasari nasi putih,7,22000,1
1,bu mau pesen 4 mie goreng djawa [SEP] noted ka...,mie goreng djawa,4,48000,2
2,4 teh obenk dong [SEP] oke kak pesanannya masu...,teh obenk,4,-1,1
3,minta 5 strawberry smoothies with ice cream ka...,strawberry smoothies with ice cream,5,11000,2
4,kak bisa pesan 6 ayam sambal ijo [SEP] noted k...,ayam sambal ijo,6,6000,1
5,kak mau order 6 nasi kotak rendang daging dong...,nasi kotak rendang daging,6,3000,1
6,beli 1 ikan kerapu goreng per ekor ya kak [SEP...,ikan kerapu goreng per ekor,1,75000,2
7,mau 2 ceker ayam ya kak [SEP] noted kak ceker ...,ceker ayam,2,60000,1
8,mbak bisa pesan 4 pancong taro dong [SEP] oke ...,pancong taro,4,20000,1
9,pak mau pesen 3 combo ayam bakar dong [SEP] co...,combo ayam bakar,3,54000,1


In [12]:
# cek tipe data per kolom
df_synthetic.info()

<class 'pandas.DataFrame'>
RangeIndex: 10050 entries, 0 to 10049
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   input_text    10050 non-null  str  
 1   product       10050 non-null  str  
 2   quantity      10050 non-null  int64
 3   price_satuan  10050 non-null  int64
 4   pattern       10050 non-null  int64
dtypes: int64(3), str(2)
memory usage: 392.7 KB


In [13]:
# cek nilai kosong
df_synthetic.isnull().sum()

input_text      0
product         0
quantity        0
price_satuan    0
pattern         0
dtype: int64

In [14]:
# cek distribusi kolom pattern
# memastikan proporsi ketiga pola sudah sesuai kesepakatan
print("Distribusi pattern:")
print(df_synthetic['pattern'].value_counts())
print(f"\nProporsi pattern:")
print(df_synthetic['pattern'].value_counts(normalize=True).round(4))

Distribusi pattern:
pattern
1    3922
2    3888
3    2240
Name: count, dtype: int64

Proporsi pattern:
pattern
1    0.3902
2    0.3869
3    0.2229
Name: proportion, dtype: float64


In [15]:
# cek baris tanpa harga (price_satuan = -1)
n_null_price = (df_synthetic['price_satuan'] == -1).sum()
total = len(df_synthetic)

print(f"Baris dengan price_satuan = -1 : {n_null_price} ({n_null_price/total:.1%})")
print(f"Baris dengan price_satuan valid: {total - n_null_price} ({(total-n_null_price)/total:.1%})")

Baris dengan price_satuan = -1 : 1750 (17.4%)
Baris dengan price_satuan valid: 8300 (82.6%)


In [16]:
# temuan penting untuk hyperparameter model
n_produk_unik = df_synthetic['product'].nunique()
panjang_teks = df_synthetic['input_text'].str.split().str.len()

print(f"Jumlah produk unik di dataset sintetis: {n_produk_unik}")
print("Nilai ini menentukan NUM_PRODCUTS di arsitektur model")

print(f"\nPanjang teks (dalam token/kata):")
print(f"Minimum: {panjang_teks.min()}")
print(f"Rata-rata: {panjang_teks.mean():.1f}")
print(f"Maksimum: {panjang_teks.max()}")
print(f"Persen 95: {panjang_teks.quantile(0.95):.0f}")


Jumlah produk unik di dataset sintetis: 4975
Nilai ini menentukan NUM_PRODCUTS di arsitektur model

Panjang teks (dalam token/kata):
Minimum: 3
Rata-rata: 16.7
Maksimum: 26
Persen 95: 22


## 5. Arsitektur Model dan Verifikasi

### Hyperparameter Berdasarkan Eksplorasi Dataset

In [17]:
# jumlah kata unik yang dikenali model
VOCAB_SIZE = 10000

# nilai persentil 95 yang didapatkan saat eksplorasi dataset
# nilainya 22, namun menggunakan nilai sedikit di atasnya sebagai buffer
MAX_SEQ_LEN = 30

# dimensi vektor representasi setiap kata
EMBEDDING_DIM = 128

# ukuran hidden layer satu arah LSTM
# output bidirectional = LSTM_UNITS x 2 = 128
LSTM_UNITS = 64

# nilai produk unik dari eksplorasi dataset sintetis
NUM_PRODUCTS = n_produk_unik
print(f"NUM_PRODUCTS (dari dataset): {NUM_PRODUCTS}")

NUM_PRODUCTS (dari dataset): 4975


### Custom Loss Function - MaskedPriceLoss

Custom Loss Function untuk output head 'price_satuan'.

Masalah yang diselesaikan:
Dataset ChatKasir memiliki baris dengan price_satuan = -1 (harga tidak disebutkan di chat). Loss function standar MSE akan menghukum model untuk kasus yang tidak memiliki jawaban valid.

Solusi:
Hitung loss HANYA untuk baris yang memiliki harga valid (bukan -1). Baris dengan harga -1 dikecualikan dari perhitungan gradient.

Konsep masking terinspirasi dari Lample et al. (2016) yang mengabaikan token padding saat menghitung loss pada sequence labeling.

In [18]:
def __init__(self, name="masked_price_loss"):
    super().__init__(name=name)
    self.NULL_INDICATOR = -1.0

def call(self, y_true, y_pred):
    # buat mask 1 untuk baris dengan harga valid
    mask = tf.cast(
        tf.not_equal(y_true, self.NULL_INDICATOR),
        dtype=tf.float32
    )

    # hitung squared error semua baris
    squared_error = tf.square(y_true - y_pred)

    # terapkan mask tanpa baris null
    masked_error = squared_error * mask

    # rata-rata jumlah baris valid saja
    n_valid = tf.maximum(tf.reduce_sum(mask), 1.0)
    return tf.reduce_sum(masked_error) / n_valid

### Mendefinisikan Arsitektur Model

Membangun model ekstraksi entitas ChatKasir.

- Pendekatan: Multi-Output dengan Shared Bidirectional LSTM Encoder.
- Referensi: Lample et al. (2016) - Neural Architectures for NER.

Returns:

tf.keras.Model dengan tiga output:
- product  : softmax (klasifikasi nama produk)
- quantity : relu (regresi jumlah pesanan)
- price    : relu (regresi harga satuan (dinormalisasi ÷1000))

In [19]:
def build_model(vocab_size=VOCAB_SIZE, max_seq_len=MAX_SEQ_LEN, embedding_dim=EMBEDDING_DIM, lstm_units=LSTM_UNITS, num_products=NUM_PRODUCTS):

    # --input--
    inputs = layers.Input(shape=(max_seq_len,), name="input_tokens")

    # --shared encoder--
    # embedding: ubah integer token menjadi vektor bermakna berdimensi 128
    # mask_zero=True token padding (0) diabaikan dalam komputasi LSTM
    x = layers.Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        mask_zero=True,
        name="embedding"
    )(inputs)

    # bidirectional LSTM (baca konteks dari kiri ke kanan dan sebaliknya)
    # return_sequences=False ambil hanya state akhir (ringkasan kalimat)
    x = layers.Bidirectional(
        layers.LSTM(lstm_units, return_sequences=False),
        name="bi_lstm"
    )(x)

    # dropout 30% neuron acak saat training, mencegah overfitting
    x = layers.Dropout(0.3, name="dropout")(x)

    # shared dense sebelum output terpecah ke 3 head
    x = layers.Dense(128, activation="relu", name="shared_dense")(x)

    # --output head--
    # head 1: product 
    product_output = layers.Dense(
        num_products, activation="softmax", name="product"
    )(x)

    # head 2: quantity (selalu positif)
    quantity_output = layers.Dense(
        1, activation="relu", name="quantity"
    )(x)

    # head 3: price (dinormalisasi /1000)
    price_output = layers.Dense(
        1, activation="relu", name="price"
    )(x)

    # --model---
    model = tf.keras.Model(
        inputs=inputs,
        outputs={
            "product": product_output,
            "quantity": quantity_output,
            "price": price_output
        },
        name="chatkasir_extractor_v0"
    )
    return model

# instansiasi & tampilkan ringkasan arsitektur
model = build_model()
model.summary()

Model: "chatkasir_extractor_v0"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_tokens        │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 30, 128)   │  1,280,000 │ input_tokens[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 30)        │          0 │ input_tokens[0][… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bi_lstm             │ (None, 128)       │     98,816 │ embedding[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ bi_lstm[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dense        │ (None, 128)       │     16,512 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ price (Dense)       │ (None, 1)         │        129 │ shared_dense[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ product (Dense)     │ (None, 4975)      │    641,775 │ shared_dense[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ quantity (Dense)    │ (None, 1)         │        129 │ shared_dense[0][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,037,361 (7.77 MB)

 Trainable params: 2,037,361 (7.77 MB)

 Non-trainable params: 0 (0.00 B)